# LightOnOCR-2 Magyar Fine-tuning (v7)

**Runtime → Change runtime type → T4 GPU**

- Javított font kezelés (rendszer fontok + ellenőrzés)
- Augmentációk: zaj, forgatás, torzítás, elmosás

In [ ]:
# 1. Telepítés + Font telepítés
!pip install -q transformers>=4.45.0 peft datasets accelerate pillow opencv-python-headless

# Magyar karaktereket támogató fontok telepítése
!apt-get install -qq fonts-dejavu fonts-liberation fonts-freefont-ttf fonts-noto-core
print('Fontok telepítve')
!fc-list :lang=hu | head -10

In [ ]:
# 2. Működő fontok keresése
import subprocess
from pathlib import Path
from PIL import Image, ImageDraw, ImageFont
import numpy as np

TEST_CHARS = 'őűŐŰ'

def font_supports_hungarian(font_path, size=24):
    try:
        font = ImageFont.truetype(font_path, size)
        img = Image.new('RGB', (100, 40), 'white')
        draw = ImageDraw.Draw(img)
        draw.text((5, 5), TEST_CHARS, fill='black', font=font)
        arr = np.array(img)
        black_pixels = np.sum(arr < 128)
        return black_pixels > 100
    except:
        return False

result = subprocess.run(['fc-list', '--format=%{file}\\n'], capture_output=True, text=True)
all_fonts = [f.strip() for f in result.stdout.split('\\n') if f.strip().endswith('.ttf')]

print(f'Összes TTF font: {len(all_fonts)}')
print('Magyar karaktereket támogató fontok:')

FONTS = []
for font_path in sorted(set(all_fonts)):
    if font_supports_hungarian(font_path):
        name = Path(font_path).stem
        FONTS.append((name, font_path))
        print(f'  ✓ {name}')

print(f'\\n=== {len(FONTS)} font működik ===')

In [ ]:
# 3. Font teszt
from IPython.display import display

print('Font teszt - ellenőrizd az ékezeteket:\\n')
for name, path in FONTS[:8]:
    try:
        font = ImageFont.truetype(path, 28)
        img = Image.new('RGB', (500, 50), 'white')
        draw = ImageDraw.Draw(img)
        draw.text((10, 10), f'{name}: őűŐŰ öüóőúéáűí', fill='black', font=font)
        display(img)
    except:
        pass

print('\\n↑ Ha látod az ékezeteket, a fontok működnek!')

In [ ]:
# 4. Augmentációk
import cv2
import random
from PIL import ImageFilter

def add_noise(img, intensity=0.02):
    arr = np.array(img).astype(np.float32)
    noise = np.random.normal(0, intensity * 255, arr.shape)
    arr = np.clip(arr + noise, 0, 255).astype(np.uint8)
    return Image.fromarray(arr)

def add_salt_pepper(img, amount=0.005):
    arr = np.array(img)
    salt = np.random.random(arr.shape[:2]) < amount/2
    arr[salt] = 255
    pepper = np.random.random(arr.shape[:2]) < amount/2
    arr[pepper] = 0
    return Image.fromarray(arr)

def rotate_image(img, max_angle=2.0):
    angle = random.uniform(-max_angle, max_angle)
    return img.rotate(angle, fillcolor='white', expand=False)

def perspective_transform(img, intensity=0.02):
    arr = np.array(img)
    h, w = arr.shape[:2]
    src = np.float32([[0,0], [w,0], [w,h], [0,h]])
    offset = int(min(w,h) * intensity)
    dst = np.float32([
        [random.randint(0, offset), random.randint(0, offset)],
        [w - random.randint(0, offset), random.randint(0, offset)],
        [w - random.randint(0, offset), h - random.randint(0, offset)],
        [random.randint(0, offset), h - random.randint(0, offset)]
    ])
    M = cv2.getPerspectiveTransform(src, dst)
    result = cv2.warpPerspective(arr, M, (w, h), borderValue=(255, 255, 255))
    return Image.fromarray(result)

def add_blur(img, radius=0.5):
    return img.filter(ImageFilter.GaussianBlur(radius=radius))

def apply_augmentations(img):
    augs = [
        (0.3, lambda x: add_noise(x, random.uniform(0.01, 0.03))),
        (0.2, lambda x: add_salt_pepper(x, random.uniform(0.002, 0.008))),
        (0.4, lambda x: rotate_image(x, random.uniform(0.5, 2.5))),
        (0.2, lambda x: perspective_transform(x, random.uniform(0.01, 0.03))),
        (0.2, lambda x: add_blur(x, random.uniform(0.3, 0.8))),
    ]
    for prob, aug_func in augs:
        if random.random() < prob:
            img = aug_func(img)
    return img

print('✓ Augmentációk definiálva')

In [ ]:
# 5. Adatgenerálás
import json

HUNGARIAN_WORDS = [
    'őr', 'őriz', 'ők', 'ősz', 'ősi', 'őszinte', 'őrült',
    'erő', 'idő', 'mező', 'tető', 'fő', 'nő', 'bő', 'hő',
    'belső', 'külső', 'felső', 'alsó', 'utolsó', 'első',
    'költő', 'festő', 'vezető', 'börtön', 'könyv', 'között',
    'Győr', 'dőlt', 'dől', 'töröl', 'pörög', 'görög', 'örök',
    'űr', 'űrlap', 'gyűrű', 'tűz', 'fűz', 'gyűjt', 'gyűlés',
    'tűnik', 'fűszer', 'hűtő', 'hűvös', 'hűség',
    'szürke', 'szűk', 'szűr', 'sűrű', 'bűvös', 'működik', 'műszer',
    'Csatornadíj', 'vízdíj', 'díj', 'tükörfúrógép', 'árvíztűrő',
    'halványszürke', 'fizetendő', 'összeg', 'összesen',
    'adószám', 'cégjegyzékszám', 'azonosító', 'határidő',
]

def gen_text():
    lines = []
    lines.append(' '.join(random.sample(HUNGARIAN_WORDS, random.randint(5, 8))))
    lines.append(f'Fizetendő összeg: {random.randint(1,99)} {random.randint(100,999):03d} Ft')
    lines.append(f'Csatornadíj: {random.randint(1,9)} {random.randint(100,999):03d} Ft')
    lines.append(f'Adószám: {random.randint(10000000,99999999)}-{random.randint(1,2)}-{random.randint(10,99)}')
    lines.append('öüóőúéáűí - ÖÜÓŐÚÉÁŰÍ')
    lines.append('Árvíztűrő tükörfúrógép')
    return '\\n'.join(lines)

def render(text, font_path, font_size=24):
    font = ImageFont.truetype(font_path, font_size)
    lines = text.split('\\n')
    h = len(lines) * int(font_size * 1.5) + 80
    bg = random.choice(['white', '#fafafa', '#f5f5f5', '#fffef0'])
    img = Image.new('RGB', (850, h), bg)
    draw = ImageDraw.Draw(img)
    y = 40
    for line in lines:
        draw.text((40, y), line, fill='black', font=font)
        y += int(font_size * 1.5)
    return img

if len(FONTS) < 3:
    print('HIBA: Kevés működő font!')
else:
    Path('training_data/images').mkdir(parents=True, exist_ok=True)
    annotations = []
    NUM_SAMPLES = 800
    AUG_RATIO = 0.6
    
    print(f'Generálás: {NUM_SAMPLES} kép, {len(FONTS)} fonttal...')
    for i in range(NUM_SAMPLES):
        text = gen_text()
        font_name, font_path = random.choice(FONTS)
        font_size = random.choice([18, 20, 22, 24, 26, 28])
        img = render(text, font_path, font_size)
        
        augmented = random.random() < AUG_RATIO
        if augmented:
            img = apply_augmentations(img)
        
        img.save(f'training_data/images/{i:05d}.png')
        annotations.append({'image': f'{i:05d}.png', 'text': text, 'font': font_name, 'augmented': augmented})
        
        if (i+1) % 100 == 0:
            print(f'  {i+1}/{NUM_SAMPLES}')
    
    with open('training_data/annotations.jsonl', 'w', encoding='utf-8') as f:
        for a in annotations:
            f.write(json.dumps(a, ensure_ascii=False) + '\\n')
    
    print(f'\\n✓ {NUM_SAMPLES} kép generálva')

In [ ]:
# 6. Példák
print('Példák:')
for i in [0, 100, 200]:
    print(f'\\nKép #{i}:')
    display(Image.open(f'training_data/images/{i:05d}.png'))

In [ ]:
# 7. Modell
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText
from peft import LoraConfig, get_peft_model

MODEL_ID = 'lightonai/LightOnOCR-2-1B-base'
print(f'Modell betöltése: {MODEL_ID}')

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, device_map='auto'
)
processor = AutoProcessor.from_pretrained(MODEL_ID)

lora_config = LoraConfig(
    r=16, lora_alpha=32,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
    lora_dropout=0.05
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# 8. Dataset
from datasets import Dataset

def load_data():
    data = []
    with open('training_data/annotations.jsonl', encoding='utf-8') as f:
        for line in f:
            e = json.loads(line)
            data.append({'image_path': f"training_data/images/{e['image']}", 'text': e['text']})
    return Dataset.from_list(data)

def process(ex):
    img = Image.open(ex['image_path']).convert('RGB')
    img_in = processor.image_processor(img, return_tensors='pt')
    txt_in = processor.tokenizer(ex['text'], return_tensors='pt', padding='max_length', max_length=512, truncation=True)
    return {
        'pixel_values': img_in['pixel_values'].squeeze(0),
        'input_ids': txt_in['input_ids'].squeeze(0),
        'attention_mask': txt_in['attention_mask'].squeeze(0),
        'labels': txt_in['input_ids'].squeeze(0),
    }

dataset = load_data().map(process, remove_columns=['image_path', 'text'])
print(f'✓ Dataset: {len(dataset)} kép')

In [ ]:
# 9. Training
from transformers import TrainingArguments, Trainer

args = TrainingArguments(
    output_dir='./lighton-hun-lora',
    num_train_epochs=5,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=5e-5,
    warmup_ratio=0.1,
    logging_steps=25,
    save_steps=150,
    bf16=True,
    remove_unused_columns=False,
    report_to='none',
)

trainer = Trainer(model=model, args=args, train_dataset=dataset)
print(f'Tanítás: {len(dataset)} kép, 5 epoch')
trainer.train()
print('\\n✓ Tanítás kész!')

In [ ]:
# 10. Mentés
print('Mentés...')
model.save_pretrained('./lighton-hun-lora')
merged = model.merge_and_unload()
merged.save_pretrained('./lighton-hun-merged')
processor.save_pretrained('./lighton-hun-merged')
print('✓ Mentve: ./lighton-hun-merged')

In [ ]:
# 11. Teszt
print('Teszt:')
for idx in [0, 200, 400]:
    img = Image.open(f'training_data/images/{idx:05d}.png')
    inputs = processor.image_processor(img, return_tensors='pt')
    inputs = {k: v.to(merged.device) for k, v in inputs.items()}
    inputs['input_ids'] = processor.tokenizer('', return_tensors='pt')['input_ids'].to(merged.device)
    
    with torch.no_grad():
        out = merged.generate(**inputs, max_new_tokens=400, do_sample=False)
    result = processor.tokenizer.decode(out[0], skip_special_tokens=True)
    
    print(f'\\n=== #{idx} ===')
    display(img)
    print(result[:300])

In [ ]:
# 12. Letöltés
!zip -r lighton-hun-merged.zip lighton-hun-merged/

from google.colab import files
files.download('lighton-hun-merged.zip')

print('\\n' + '='*50)
print('MAC-EN:')
print('='*50)
print('unzip lighton-hun-merged.zip')
print('mlx_vlm convert --hf-path lighton-hun-merged \\\\')
print('    --mlx-path models/lighton-hun-mlx -q --q-bits 4')